# ETL Processes
Use this notebook to develop the ETL process for each of your tables before completing the `etl.py` file to load the whole datasets.

In [2]:
# Libraries
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from sql_queries import *
import os
import glob
import math

In [4]:
# DATA:
def get_files(filepath):
    all_files = []
    for root, dirs, files in os.walk(filepath):
        files = glob.glob(os.path.join(root,'*.json'))
        for f in files :
            all_files.append(os.path.abspath(f))
    
    return all_files

In [6]:
df = pd.read_json(get_files('data/song_data')[0], lines=True)
df.head()

,num_songs,artist_id,artist_latitude,artist_longitude,artist_location,artist_name,song_id,title,duration,year
0,1,ARD7TVE1187B99BFB1,NaN,NaN,California - LA,Casual,SOMZWCG12A8C13C480,I Didn't Mean To,218.93179,0


In [7]:
df = pd.read_json(get_files('data/log_data')[0], lines=True)
df.head()

,artist,auth,firstName,gender,itemInSession,lastName,length,level,location,method,page,registration,sessionId,song,status,ts,userAgent,userId
0,NaN,Logged In,Walter,M,0,Frye,NaN,free,"San Francisco-Oakland-Hayward, CA",GET,Home,1540919166796,38,NaN,200,1541105830796,"""Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_4...",39
1,NaN,Logged In,Kaylee,F,0,Summers,NaN,free,"Phoenix-Mesa-Scottsdale, AZ",GET,Home,1540344794796,139,NaN,200,1541106106796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
2,Des'ree,Logged In,Kaylee,F,1,Summers,246.30812,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,You Gotta Be,200,1541106106796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
3,NaN,Logged In,Kaylee,F,2,Summers,NaN,free,"Phoenix-Mesa-Scottsdale, AZ",GET,Upgrade,1540344794796,139,NaN,200,1541106132796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
4,Mr Oizo,Logged In,Kaylee,F,3,Summers,144.03873,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,Flat 55,200,1541106352796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8


In [8]:

def process_song_file(cur, filepath):
    """processing the song files and getting data from it and insert it in songs and artists tables"""
    # open song file
    df = pd.read_json(filepath, lines=True)

    # insert song record
    song_data = [
        df.song_id[0],
        df.title[0],
        df.artist_id[0],
        int(df.year[0]),
        float(df.duration[0])
    ]
    cur.execute(song_table_insert, song_data)

    # Insert artist record
    artist_data = [
        df.artist_id[0],
        df.artist_name[0],
        df.artist_location[0],
        None if math.isnan(float(df.artist_latitude[0])) else float(df.artist_latitude[0]),
        None if math.isnan(float(df.artist_longitude[0])) else float(df.artist_longitude[0])
    ]
    cur.execute(artist_table_insert, artist_data)
    

def process_log_file(cur, filepath):
    """processing the log files and getting data from it to insert in time, users and songplays tables"""
    # open log file
    df = pd.read_json(filepath, lines=True)
    
    # filter by NextSong action
    df = df[df['page'] == 'NextSong']

    # convert timestamp column to datetime
    df['TS'] = pd.to_datetime(df['ts'], unit='ms')
    
    # insert time data records
    time_df = pd.DataFrame({
        'TS': df['TS'],
        'Hour': df['TS'].dt.hour,
        'Day': df['TS'].dt.day,
        'WeekOfYear': df['TS'].dt.isocalendar().week,
        'Month': df['TS'].dt.month,
        'Year': df['TS'].dt.year,
        'WeekDay': df['TS'].dt.dayofweek
    }).drop_duplicates(subset=['TS'])

    for i, row in time_df.iterrows():
        cur.execute(time_table_insert, list(row))

    # load user table
    user_df = df[['userId', 'firstName', 'lastName', 'gender', 'level']].drop_duplicates(subset=['userId'])
    
    # insert user records
    for i, row in user_df.iterrows():
        cur.execute(user_table_insert, list(row))
        
    # insert songplay records
    for index, row in df.iterrows():
        # get songid and artistid from songs and artists tables
        SongIDD = row.song.replace('\'', '\'\'')
        ArtistIDD = row.artist.replace('\'', '\'\'')
        
        Query_artist_song = song_select.format(SongIDD, ArtistIDD)
        cur.execute(Query_artist_song)
        results = cur.fetchone()
        if results:
            songid, artistid = results
        else:
            songid, artistid = None , None
        
        songplay_data = (row.TS, row.userId, row.level, songid, 
                             artistid, row.sessionId, row.location, row.userAgent)
        cur.execute(songplay_table_insert, songplay_data)

                


def process_data(cur, conn, filepath, func):
    """getting all the json files from the filepath and aplly funcs on every fill to fill the tables """
    # get all files matching extension from directory
    all_files = []
    for root, dirs, files in os.walk(filepath):
        files = glob.glob(os.path.join(root,'*.json'))
        for f in files :
            all_files.append(os.path.abspath(f))

    # get total number of files found
    num_files = len(all_files)
    print('{} files found in {}'.format(num_files, filepath))

    # iterate over files and process
    for i, datafile in enumerate(all_files, 1):
        func(cur, datafile)
        conn.commit()
        print('{}/{} files processed.'.format(i, num_files))



    
conn = pyodbc.connect(
'Driver={ODBC Driver 17 for SQL Server};'
'Server=DESKTOP-TAPLSFS\SQLEXPRESS;'
'Database=DataModellingWithSQL;'
'Trusted_Connection=yes;'
)
cur = conn.cursor()


process_data(cur, conn, filepath='Data/song_data', func=process_song_file)
process_data(cur, conn, filepath='Data/log_data', func=process_log_file)

cur.close()
conn.close()


<>:104: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:104: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\Omar Nasr\AppData\Local\Temp\ipykernel_38780\3103755192.py:104: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  'Server=DESKTOP-TAPLSFS\SQLEXPRESS;'


71 files found in Data/song_data
1/71 files processed.
2/71 files processed.
3/71 files processed.
4/71 files processed.
5/71 files processed.
6/71 files processed.
7/71 files processed.
8/71 files processed.
9/71 files processed.
10/71 files processed.
11/71 files processed.
12/71 files processed.
13/71 files processed.
14/71 files processed.
15/71 files processed.
16/71 files processed.
17/71 files processed.
18/71 files processed.
19/71 files processed.
20/71 files processed.
21/71 files processed.
22/71 files processed.
23/71 files processed.
24/71 files processed.
25/71 files processed.
26/71 files processed.
27/71 files processed.
28/71 files processed.
29/71 files processed.
30/71 files processed.
31/71 files processed.
32/71 files processed.
33/71 files processed.
34/71 files processed.
35/71 files processed.
36/71 files processed.
37/71 files processed.
38/71 files processed.
39/71 files processed.
40/71 files processed.
41/71 files processed.
42/71 files processed.
43/71 file